In [ ]:
!pip install dash pandas plotly

In [1]:
from google.colab import files
uploaded = files.upload()

Saving student_performance.csv to student_performance.csv


In [ ]:
!pip install dash jupyter-dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.3 MB/s eta 0:00:00


In [ ]:
!pip install pyngrok dash --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 81.5 MB/s eta 0:00:00


In [6]:
from pyngrok import ngrok
ngrok.set_auth_token("3A3h2PGzeJlDE7PsAityiqBJBvo_3qzUpxZSSWXKg5KEjvDkh")

In [ ]:
# Install packages
!pip install dash plotly pandas pyngrok --quiet

import pandas as pd
import dash
from dash import dcc, html, dash_table
import plotly.express as px
import plotly.graph_objs as go
from dash.dependencies import Input, Output

# ---------------- LOAD DATA ----------------
df = pd.read_csv("student_performance.csv")

# Rename columns
df.rename(columns={
    'student_id': 'Name',
    'total_score': 'Marks',
    'attendance_percentage': 'Attendance(%)',
    'class_participation': 'Logins'
}, inplace=True)

# Make names readable
df['Name'] = df['Name'].apply(lambda x: f"Student {x}")

# Clean data
df.dropna(subset=["Name"], inplace=True)
df['Marks'] = df['Marks'].fillna(df['Marks'].mean())
df['Attendance(%)'] = df['Attendance(%)'].fillna(df['Attendance(%)'].mean())
df['Logins'] = df['Logins'].fillna(df['Logins'].mean())

# ---------------- STATUS ----------------
def get_status(row):
    if row['Marks'] < 50 or row['Attendance(%)'] < 60:
        return 'Critical'
    elif row['Marks'] < 65 or row['Attendance(%)'] < 70:
        return 'At Risk'
    else:
        return 'On Track'

df['Status'] = df.apply(get_status, axis=1)
# ---------------- MACHINE LEARNING ----------------
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Features & Target
X = df[['weekly_self_study_hours', 'Attendance(%)', 'Logins']]
y = df['Status']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Model
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

# Accuracy
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# Predict for dataset
df['Predicted_Status'] = model.predict(X)

# Sample for performance
df_sample = df.sample(8000, random_state=42).copy()

color_map = {
    'On Track': '#22c55e',
    'At Risk': '#f59e0b',
    'Critical': '#ef4444'
}

# ---------------- INSIGHTS ----------------
def generate_insights(df):
    top = df.nlargest(1, 'Marks')['Name'].values[0]
    low = df.nsmallest(1, 'Marks')['Name'].values[0]
    avg_marks = round(df['Marks'].mean(), 2)

    return [
        f"🏆 Top Performer: {top}",
        f"⚠️ Needs Attention: {low}",
        f"📊 Average Marks: {avg_marks}"
    ]

# ---------------- DASH APP ----------------
app = dash.Dash(__name__)
app.title = "AI Student Analytics Dashboard"

# ---------------- LAYOUT ----------------
app.layout = html.Div(style={
    'backgroundColor': '#f8fafc',
    'color': '#1e293b',
    'fontFamily': 'Arial'
}, children=[

    html.H1("🎓 AI-Powered Student Performance Dashboard",
            style={'textAlign': 'center', 'padding': '20px'}),

    # FILTERS
    html.Div([
        dcc.Dropdown(
            id='status-filter',
            options=[{'label': 'All', 'value': 'All'}] +
                    [{'label': i, 'value': i} for i in sorted(df_sample['Status'].unique())],
            value='All',
            clearable=False,
            style={'width': '200px', 'display': 'inline-block', 'marginRight': '20px'}
        ),

        dcc.Dropdown(
            id='student-filter',
            options=[{'label': i, 'value': i} for i in df_sample['Name'].unique()],
            multi=True,
            placeholder="Select Students",
            style={'width': '300px', 'display': 'inline-block'}
        )
    ], style={'padding': '20px'}),

    # KPI CARDS
    html.Div(id='kpi-cards', style={
        'display': 'flex',
        'justifyContent': 'space-around',
        'padding': '10px'
    }),

    # INSIGHTS
    html.Div(id='insights-box', style={
        'background': '#ffffff',
        'padding': '15px',
        'margin': '20px',
        'borderRadius': '10px',
        'boxShadow': '0px 2px 8px rgba(0,0,0,0.1)'
    }),

    html.H3("📊 Performance Analytics", style={'paddingLeft': '20px'}),

    html.Div([
        dcc.Graph(id='bar-chart'),
        dcc.Graph(id='pie-chart'),
    ], style={'display': 'flex'}),

    html.Div([
        dcc.Graph(id='scatter-chart'),
        dcc.Graph(id='heatmap'),
    ], style={'display': 'flex'}),

    dcc.Graph(id='marks-distribution'),
    dcc.Graph(id='study-chart'),
    dcc.Graph(id='before-after-chart'),
    dcc.Graph(id='feature-importance'),

    html.H3("📋 Personalized Recommendations", style={'padding': '20px'}),

    dash_table.DataTable(
        id='recommendation-table',
        columns=[
            {'name': 'Name', 'id': 'Name'},
            {'name': 'Marks', 'id': 'Marks'},
            {'name': 'Attendance(%)', 'id': 'Attendance(%)'},
            {'name': 'Logins', 'id': 'Logins'},
            {'name': 'Suggestion', 'id': 'Suggestion'}
        ],
        style_cell={
            'textAlign': 'center',
            'backgroundColor': 'white',
            'color': 'black',
            'border': '1px solid #e5e7eb'
        },
        style_header={
            'backgroundColor': '#e2e8f0',
            'fontWeight': 'bold',
            'color': 'black'
        },
        page_size=10
    )
])

# ---------------- CALLBACK ----------------
@app.callback(
    [Output('bar-chart', 'figure'),
     Output('before-after-chart', 'figure'),
     Output('heatmap', 'figure'),
     Output('recommendation-table', 'data'),
     Output('pie-chart', 'figure'),
     Output('scatter-chart', 'figure'),
     Output('kpi-cards', 'children'),
     Output('marks-distribution', 'figure'),
     Output('study-chart', 'figure'),
     Output('insights-box', 'children'),
     Output('feature-importance', 'figure')],
    [Input('status-filter', 'value'),
     Input('student-filter', 'value')]
)
def update_dashboard(status, selected_students):

    filtered_df = df_sample.copy()

    if status != 'All':
        filtered_df = filtered_df[filtered_df['Status'] == status]

    if selected_students:
        filtered_df = filtered_df[filtered_df['Name'].isin(selected_students)]

    if filtered_df.empty:
        return [go.Figure()] * 11

    if len(filtered_df) > 3000:
      filtered_df = filtered_df.sample(3000, random_state=42)

    # KPI
    kpis = [
        html.Div(f"👨‍🎓 Students: {len(filtered_df)}",
                 style={'background': '#ffffff', 'padding': '12px', 'borderRadius': '10px',
                        'boxShadow': '0px 2px 8px rgba(0,0,0,0.1)'}),
        html.Div(f"📊 Avg Marks: {round(filtered_df['Marks'].mean(), 2)}",
                 style={'background': '#ffffff', 'padding': '12px', 'borderRadius': '10px',
                        'boxShadow': '0px 2px 8px rgba(0,0,0,0.1)'}),
        html.Div(f"📅 Avg Attendance: {round(filtered_df['Attendance(%)'].mean(), 2)}%",
                 style={'background': '#ffffff', 'padding': '12px', 'borderRadius': '10px',
                        'boxShadow': '0px 2px 8px rgba(0,0,0,0.1)'}),
        html.Div(f"🤖 Model Accuracy: {round(accuracy*100, 2)}%",
         style={'background': '#ffffff', 'padding': '12px', 'borderRadius': '10px',
                'boxShadow': '0px 2px 8px rgba(0,0,0,0.1)'}),
    ]

    # Common style
    def style_fig(fig, title):
        fig.update_layout(
            title=title,
            paper_bgcolor='white',
            plot_bgcolor='white',
            font=dict(color='black'),
            title_font=dict(size=18),
            xaxis=dict(showgrid=True, gridcolor='#e5e7eb'),
            yaxis=dict(showgrid=True, gridcolor='#e5e7eb')
        )
        return fig

    # Graphs
    bar_fig = style_fig(px.bar(filtered_df.head(20), x='Name', y='Marks', color='Status'),
                        "🏆 Top Students")

    pie_fig = style_fig(px.pie(filtered_df, names='Status', color='Status'),
                        "📊 Status Distribution")

    scatter_fig = style_fig(px.scatter(filtered_df.sample(min(2000, len(filtered_df))),
                                       x='Attendance(%)', y='Marks',
                                       size='Logins', color='Status'),
                            "📈 Attendance vs Marks vs Engagement")

    heatmap_fig = style_fig(px.imshow(filtered_df[['Marks', 'Attendance(%)', 'Logins']].corr(),
                                      text_auto=True),
                            "🔥 Correlation Heatmap")

    marks_dist = style_fig(px.histogram(filtered_df, x='Marks', nbins=30),
                           "📊 Marks Distribution")

    study_fig = style_fig(px.scatter(filtered_df,
                                     x='weekly_self_study_hours',
                                     y='Marks',
                                     color='Status'),
                          "📚 Study Hours vs Marks")

    # Before After
    intervention_df = filtered_df[filtered_df['Status'] != 'On Track'].head(20).copy()
    intervention_df['Improved Marks'] = intervention_df['Marks'] + 10

    before_after_fig = go.Figure()
    before_after_fig.add_trace(go.Scatter(x=intervention_df['Name'],
                                          y=intervention_df['Marks'],
                                          mode='lines+markers',
                                          name='Before'))
    before_after_fig.add_trace(go.Scatter(x=intervention_df['Name'],
                                          y=intervention_df['Improved Marks'],
                                          mode='lines+markers',
                                          name='After'))

    before_after_fig = style_fig(before_after_fig,
                                "📉 Before vs After Academic Improvement")

    # Recommendations (PERSONALIZED)
    rec_list = []
    for _, row in filtered_df.head(500).iterrows():

        if row['Status'] == 'On Track':
            continue

        suggestions = []

        if row['Marks'] < 50:
            suggestions.append("Focus on core subjects")
        if row['Attendance(%)'] < 60:
            suggestions.append("Improve attendance")
        if row['Logins'] < 3:
            suggestions.append("Increase LMS usage")
        if row['weekly_self_study_hours'] < 10:
            suggestions.append("Increase study hours")
        if 50 <= row['Marks'] < 70:
            suggestions.append("Revise weak topics")
        if row['Marks'] >= 70 and row['Attendance(%)'] < 75:
            suggestions.append("Improve consistency")

        rec_list.append({
            'Name': row['Name'],
            'Marks': round(row['Marks'], 1),
            'Attendance(%)': round(row['Attendance(%)'], 1),
            'Logins': round(row['Logins'], 1),
            'Suggestion': " | ".join(suggestions)
        })

    insights_ui = [html.P(i) for i in generate_insights(filtered_df)]
    # ---------------- FEATURE IMPORTANCE ----------------
    importance = model.feature_importances_

    feat_fig = px.bar(
        x=X.columns,
        y=importance,
        title="🤖 Feature Importance (ML Model)"
    )

    feat_fig.update_layout(
        paper_bgcolor='white',
        plot_bgcolor='white',
        font=dict(color='black')
    )
    return (bar_fig, before_after_fig, heatmap_fig, rec_list, pie_fig, scatter_fig, kpis, marks_dist, study_fig, insights_ui, feat_fig)


# ---------------- RUN ----------------
if __name__ == "__main__":
    from pyngrok import ngrok
    ngrok.kill()  # reset old sessions
    public_url = ngrok.connect(8050)
    print("🚀 Public URL:", public_url)
    app.run(port=8050)

🚀 Public URL: NgrokTunnel: "https://deloras-syringeful-tory.ngrok-free.dev" -> "http://localhost:8050"
Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 16:58:12] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 16:58:12] "GET /_dash-layout HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 16:58:12] "GET /_dash-dependencies HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 16:58:15] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 16:58:15] "GET /_dash-component-suites/dash/dcc/async-dropdown.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 16:58:15] "GET /_dash-component-suites/dash/dcc/async-graph.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 16:58:15] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 16

In [ ]:
import sys
print("google.colab" in sys.modules)

True
